# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).**

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`.

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]

    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()

    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))

    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")

    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania: 7.10s


In [ ]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]

    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))

    all_titles = [title for sublist in results for title in sublist]

    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")

    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania (wątki): 1.58s


---
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [ ]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))

print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


---
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [ ]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()

    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)

    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

ModuleNotFoundError: No module named 'lab2_functions'

---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [ ]:
import requests
import time
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor

CAT_API_URL = "https://catfact.ninja/fact"
FACTS = 20

def pobierz_fakt():
    try:
      response = requests.get(CAT_API_URL, timeout=10)
      response.raise_for_status()
      return response.json().get("fact")
    except requests.RequestException as e:
      return f"Błąd: {e}"

start_seq = time.perf_counter()

fakty_seq = []
for _ in range(FACTS):
  fakty_seq.append(pobierz_fakt())
koniec_seq = time.perf_counter()
czas_seq = koniec_seq - start_seq

print("=== FAKTY POBRANE SEKWENCYJNIE ===")
for i, fakt in enumerate(fakty_seq, 1):
  print(f"{i}. {fakt}")
print(f"\nCzas sekwencyjny: {czas_seq:.4f} s")

start_thread = time.perf_counter()

with ThreadPoolExecutor(max_workers=10) as executor:
  fakty_thread = list(executor.map(lambda _: pobierz_fakt(), range(FACTS)))
koniec_thread = time.perf_counter()
czas_thread = koniec_thread - start_thread

print("\n=== FAKTY POBRANE WIELOWĄTKOWO ===")
for i, fakt in enumerate(fakty_thread, 1):
  print(f"{i}. {fakt}")

print(f"\nCzas wielowątkowy: {czas_thread:.4f} s")

print("\n=== PORÓWNANIE ===")
print(f"Sekwencyjnie:   {czas_seq:.4f} s")
print(f"Wielowątkowo:   {czas_thread:.4f} s")

if czas_thread < czas_seq:
  print("Wielowątkowo działa szybciej.")
elif czas_thread > czas_seq:
  print("Sekwencyjnie działa szybciej.")
else:
  print("Czasy są takie same.")


=== FAKTY POBRANE SEKWENCYJNIE ===
1. Since cats are so good at hiding illness, even a single instance of a symptom should be taken very seriously.
2. A cat can jump up to five times its own height in a single bound.
3. Approximately 40,000 people are bitten by cats in the U.S. annually.
4. In Siam, the cat was so revered that one rode in a chariot at the head of a parade celebrating the new king.
5. An adult lion's roar can be heard up to five miles (eight kilometers) away.
6. A cat has approximately 60 to 80 million olfactory cells (a human has between 5 and 20 million).
7. Unlike other cats, lions have a tuft of hair at the end of their tails.
8. Cats' hearing is much more sensitive than humans and dogs.
9. There are more than 500 million domestic cats in the world, with approximately 40 recognized breeds.
10. The average lifespan of an outdoor-only (feral and non-feral) is about 3 years; an indoor-only cat can live 16 years and longer. Some cats have been documented to have a longe

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [13]:
import queue
import threading
import time

q_parzyste = queue.Queue()
q_nieparzyste = queue.Queue()

LICZBA_ELEMENTOW = 20
STOP = None


def producent():
    for i in range(1, LICZBA_ELEMENTOW + 1):
        if i % 2 == 0:
            q_parzyste.put(i)
            print(f"Producent dodał do kolejki parzystej: {i}", flush=True)
        else:
            q_nieparzyste.put(i)
            print(f"Producent dodał do kolejki nieparzystej: {i}", flush=True)

        time.sleep(0.05)

    q_parzyste.put(STOP)
    q_nieparzyste.put(STOP)


def konsument_parzyste():
    while True:
        liczba = q_parzyste.get()

        if liczba is STOP:
            q_parzyste.task_done()
            break

        print(f"\nKonsument parzysty pobrał: {liczba}", flush=True)
        q_parzyste.task_done()
        time.sleep(0.05)


def konsument_nieparzyste():
    while True:
        liczba = q_nieparzyste.get()

        if liczba is STOP:
            q_nieparzyste.task_done()
            break

        print(f"\nKonsument nieparzysty pobrał: {liczba}", flush=True)
        q_nieparzyste.task_done()
        time.sleep(0.05)


watek_producent = threading.Thread(target=producent)
watek_parzyste = threading.Thread(target=konsument_parzyste)
watek_nieparzyste = threading.Thread(target=konsument_nieparzyste)

watek_producent.start()
watek_parzyste.start()
watek_nieparzyste.start()

watek_producent.join()
watek_parzyste.join()
watek_nieparzyste.join()

print("Program zakończył działanie.", flush=True)

Producent dodał do kolejki nieparzystej: 1

Konsument nieparzysty pobrał: 1
Producent dodał do kolejki parzystej: 2
Konsument parzysty pobrał: 2

Producent dodał do kolejki nieparzystej: 3
Konsument nieparzysty pobrał: 3

Producent dodał do kolejki parzystej: 4
Konsument parzysty pobrał: 4

Producent dodał do kolejki nieparzystej: 5

Konsument nieparzysty pobrał: 5
Producent dodał do kolejki parzystej: 6
Konsument parzysty pobrał: 6

Producent dodał do kolejki nieparzystej: 7
Konsument nieparzysty pobrał: 7

Producent dodał do kolejki parzystej: 8

Konsument parzysty pobrał: 8
Producent dodał do kolejki nieparzystej: 9
Konsument nieparzysty pobrał: 9

Producent dodał do kolejki parzystej: 10
Konsument parzysty pobrał: 10

Producent dodał do kolejki nieparzystej: 11

Konsument nieparzysty pobrał: 11
Producent dodał do kolejki parzystej: 12
Konsument parzysty pobrał: 12

Producent dodał do kolejki nieparzystej: 13
Konsument nieparzysty pobrał: 13

Producent dodał do kolejki parzystej: 14

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [15]:
import multiprocessing
import time

def calculate_power_sum(n):
    suma = 0
    for i in range(1, 101):
        suma += n ** i
    return suma

if __name__ == "__main__":
  numbers = list(range(1, 10001))

  start_time = time.perf_counter()

  with multiprocessing.Pool() as pool:
    results = pool.map(calculate_power_sum, numbers)
  end_time = time.perf_counter()

  print("Pierwsze 10 wyników:")
  for i in range(10):
    print(f"{numbers[i]} -> {results[i]}")

  print(f"\nCzas wykonania: {end_time - start_time:.4f}s")
  print(f"\nLiczba obliczonych wyników: {len(results)}")

Pierwsze 10 wyników:
1 -> 100
2 -> 2535301200456458802993406410750
3 -> 773066281098016996554691694648431909053161283000
4 -> 2142584059011987034055949456454883470029603991710390447068500
5 -> 9860761315262647567646607066034827870915080438862787559628486633300780
6 -> 783982348200085087316028320589669384644572452567545845851686359643396569772850
7 -> 3773555927895550989902089063950252946000070398722062967756211219956973369576416070000
8 -> 2328041115810841241449652215325003612630249592761070000727017656405007199729527664209597000
9 -> 298815737485359115506128987290252080182887634235068807971396831956479052263964955868682786424500
10 -> 11111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111110

Czas wykonania: 0.3540s

Liczba obliczonych wyników: 10000
